# Feature Engineering

In [145]:
import polars as pl
from pathlib import Path
import tomllib
import os

In [146]:
config_file_path = Path("config.toml")

with config_file_path.open("rb") as config_file:
    config = tomllib.load(config_file)

COMPANIES = config['companies']

In [147]:
company = next(company for company in COMPANIES)

In [148]:
lf = pl.scan_parquet(f'ETL/data/enriched/parquet/{company}_share_prices_enriched.parquet')

In [149]:
lf = lf.with_columns(
    # Log Return
    (pl.col('Adj. Close') / pl.col('Adj. Close').shift(1)).log().alias(f'Log Return')
)

for window in [5, 10, 20]:
    lf = lf.with_columns(
        # Rolling Log Returns
        (pl.col('Adj. Close') / pl.col('Adj. Close').shift(window)).log().alias(f'Log Return {window}d'),

        # Volatility
        pl.col('Log Return').rolling_std(window).log1p().alias(f'Volatility {window}d'),

        # Moving Averages
        pl.col("Adj. Close").rolling_mean(5).alias(f"Moving Average {window}d"),

        # Momentum Pct. Change
        ((pl.col("Adj. Close") / pl.col("Adj. Close").shift(window)) - 1).alias(f"Momentum Pct. {window}d"),

        # Log Volume Ratio
        ((pl.col("Volume") / pl.col("Volume").rolling_mean(window))).log().alias(f"Log Volume Ratio {window}d"),
    )

    lf = lf.with_columns(
        # Log MA Ratio
        (pl.col("Adj. Close") / pl.col(f"Moving Average {window}d")).log().alias(f"Log MA Ratio {window}d")
    )

lf = lf.with_columns(
    # Intraday Returns
    ((pl.col('Close') / pl.col('Open')) - 1).alias('Intraday Pct. Return'),

    # Ranges
    (pl.col('High') - pl.col('Low')).alias('Range'),
    ((pl.col('High') - pl.col('Low')) / pl.col('Close')).alias('Range Pct.'),

    # Close Position
    (((pl.col('Close') - pl.col('Low')) / (pl.col('High') - pl.col('Low'))) - 0.5).alias('Close Position'),

    # Log Volume Change
    (pl.col("Volume") / pl.col("Volume").shift(1)).log().tanh().alias("Log Volume Change"),

    # Log Market Cap
    (pl.col("Adj. Close") * pl.col("Shares Outstanding")).log().alias("Log Market Cap"),

    # Dilution / Issuance
    (pl.col("Shares Outstanding") / pl.col("Shares Outstanding").shift(1) - 1).alias('Delta Pct. Dilution / Issuance'),

    # Volume Return Interaction
    (pl.col("Log Return") * pl.col("Log Volume Ratio 5d")).tanh().alias("Interaction Return Volume 5d"),
    (pl.col("Log Return") * pl.col("Log Volume Ratio 10d")).tanh().alias("Interaction Return Volume 10d"),
    (pl.col("Log Return") * pl.col("Log Volume Ratio 20d")).tanh().alias("Interaction Return Volume 20d"),

    # Volume Volatility Interaction
    (pl.col("Volatility 5d") * pl.col("Log Volume Ratio 5d")).tanh().alias("Interaction Volatility Volume 5d"),
    (pl.col("Volatility 10d") * pl.col("Log Volume Ratio 10d")).tanh().alias("Interaction Volatility Volume 10d"),
    (pl.col("Volatility 20d") * pl.col("Log Volume Ratio 20d")).tanh().alias("Interaction Volatility Volume 20d"),

    # Momentum Volatility Interaction
    (pl.col("Momentum Pct. 5d") * pl.col("Volatility 5d")).tanh().alias("Interaction Momentum Volatility 5d"),
    (pl.col("Momentum Pct. 10d") * pl.col("Volatility 10d")).tanh().alias("Interaction Momentum Volume 10d"),
    (pl.col("Momentum Pct. 20d") * pl.col("Volatility 20d")).tanh().alias("Interaction Momentum Volume 20d"),

    # Target Engineering
    (pl.col("Adj. Close").shift(-1) / pl.col("Adj. Close") - 1).alias("Target Return Metric"),
    ((pl.col("Adj. Close").shift(-1) / pl.col("Adj. Close") - 1) > 0).alias("Target Return Class")
)

In [150]:
with pl.Config(tbl_rows=-1):
    display(lf.head(21).collect())

Open,High,Low,Close,Adj. Close,Volume,Shares Outstanding,Log Return,Log Return 5d,Volatility 5d,Moving Average 5d,Momentum Pct. 5d,Log Volume Ratio 5d,Log MA Ratio 5d,Log Return 10d,Volatility 10d,Moving Average 10d,Momentum Pct. 10d,Log Volume Ratio 10d,Log MA Ratio 10d,Log Return 20d,Volatility 20d,Moving Average 20d,Momentum Pct. 20d,Log Volume Ratio 20d,Log MA Ratio 20d,Intraday Pct. Return,Range,Range Pct.,Close Position,Log Volume Change,Log Market Cap,Delta Pct. Dilution / Issuance,Interaction Return Volume 5d,Interaction Return Volume 10d,Interaction Return Volume 20d,Interaction Volatility Volume 5d,Interaction Volatility Volume 10d,Interaction Volatility Volume 20d,Interaction Momentum Volatility 5d,Interaction Momentum Volume 10d,Interaction Momentum Volume 20d,Target Return Metric,Target Return Class
f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool
100.86,101.79,99.88,100.58,100.58,102279660,9.9800e9,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-0.002776,1.91,0.01899,-0.133508,null,27.634802,null,null,null,null,null,null,null,null,null,null,0.015609,true
101.05,102.2,100.56,102.15,102.15,79546260,9.9800e9,0.015489,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.010886,1.64,0.016055,0.469512,-0.246208,27.650291,0.0,null,null,null,null,null,null,null,null,null,-0.000098,false
102.22,102.65,100.88,102.14,102.14,93112340,9.9800e9,-0.000098,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-0.000783,1.77,0.017329,0.211864,0.156179,27.650193,0.0,null,null,null,null,null,null,null,null,null,0.06168,true
102.0,109.0,101.9,108.44,108.44,134334180,9.9800e9,0.059853,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.063137,7.1,0.065474,0.421127,0.350947,27.710046,0.0,null,null,null,null,null,null,null,null,null,0.05284,true
110.02,114.6,109.31,114.17,114.17,161743860,9.9800e9,0.051492,null,null,105.496,null,0.348034,0.079016,null,null,105.496,null,null,0.079016,null,null,105.496,null,null,0.079016,0.03772,5.29,0.046334,0.418715,0.183578,27.761537,0.0,0.017919,null,null,null,null,null,null,null,null,0.010598,true
112.88,116.67,112.25,115.38,115.38,137331340,9.9800e9,0.010542,0.137278,0.026186,108.456,0.147147,0.124842,0.061886,null,null,108.456,null,null,0.061886,null,null,108.456,null,null,0.061886,0.022147,4.42,0.038308,0.208145,-0.162173,27.77208,0.0,0.001316,null,null,0.003269,null,null,0.003853,null,null,0.043595,true
117.3,123.05,116.75,120.41,120.41,240764020,9.9800e9,0.042672,0.16446,0.025912,112.108,0.178757,0.450396,0.07144,null,null,112.108,null,null,0.07144,null,null,112.108,null,null,0.07144,0.026513,6.3,0.052321,0.080952,0.509031,27.814752,0.0,0.019217,null,null,0.01167,null,null,0.004632,null,null,-0.013786,false
118.62,120.0,115.8,118.75,118.75,158600200,9.9800e9,-0.013882,0.150676,0.030426,115.43,0.16262,-0.048937,0.028356,null,null,115.43,null,null,0.028356,null,null,115.43,null,null,0.028356,0.001096,4.2,0.035368,0.202381,-0.394764,27.800869,0.0,0.000679,null,null,-0.001489,null,null,0.004948,null,null,0.007832,true
119.5,122.25,119.3,119.68,119.68,115413880,9.9800e9,0.007801,0.098624,0.026532,117.678,0.103652,-0.343818,0.016869,null,null,117.678,null,null,0.016869,null,null,117.678,null,null,0.016869,0.001506,2.95,0.024649,-0.371186,-0.307572,27.80867,0.0,-0.002682,null,null,-0.009122,null,null,0.00275,null,null,-0.027323,false


In [151]:
lf = lf.drop([
    "Open", 
    "High",
    "Low",
    "Close",
    "Adj. Close",
    "Moving Average 5d",
    "Moving Average 10d",
    "Moving Average 20d",
    "Volume",
    "Range",
    "Shares Outstanding"
])

lf = lf.drop_nulls()

In [152]:
with pl.Config(tbl_rows=-1):
    display(lf.head(21).collect())

Log Return,Log Return 5d,Volatility 5d,Momentum Pct. 5d,Log Volume Ratio 5d,Log MA Ratio 5d,Log Return 10d,Volatility 10d,Momentum Pct. 10d,Log Volume Ratio 10d,Log MA Ratio 10d,Log Return 20d,Volatility 20d,Momentum Pct. 20d,Log Volume Ratio 20d,Log MA Ratio 20d,Intraday Pct. Return,Range Pct.,Close Position,Log Volume Change,Log Market Cap,Delta Pct. Dilution / Issuance,Interaction Return Volume 5d,Interaction Return Volume 10d,Interaction Return Volume 20d,Interaction Volatility Volume 5d,Interaction Volatility Volume 10d,Interaction Volatility Volume 20d,Interaction Momentum Volatility 5d,Interaction Momentum Volume 10d,Interaction Momentum Volume 20d,Target Return Metric,Target Return Class
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool
0.014307,-0.009145,0.044674,-0.009103,-0.672253,0.000936,-0.005175,0.033023,-0.005162,-0.566497,0.000936,0.155995,0.030794,0.168821,-0.67668,0.000936,0.009359,0.015907,0.334225,-0.039191,27.790349,0.0,-0.009618,-0.008105,-0.009681,-0.030023,-0.018705,-0.020835,-0.000407,-0.00017,0.005199,0.006975,true
0.006951,-0.043959,0.038868,-0.043007,-0.362231,0.016988,-0.013342,0.032708,-0.013253,-0.450038,0.016988,0.147457,0.030744,0.158884,-0.586347,0.016988,-0.003032,0.013854,0.243902,0.085379,27.7973,0.0,-0.002518,-0.003128,-0.004076,-0.014078,-0.014719,-0.018025,-0.001672,-0.000433,0.004885,0.005068,true
0.005056,0.040129,0.005621,0.040945,-0.105008,0.014033,-0.012777,0.032719,-0.012696,-0.494385,0.014033,0.152611,0.030701,0.164872,-0.630512,0.014033,0.003119,0.012691,0.248344,-0.056007,27.802355,0.0,-0.000531,-0.002499,-0.003188,-0.00059,-0.016175,-0.019355,0.00023,-0.000415,0.005062,0.012355,true
0.012279,0.03937,0.005461,0.040155,0.004242,0.018417,0.013793,0.032628,0.013889,-0.43316,0.018417,0.105038,0.028296,0.110752,-0.586599,0.018417,0.014487,0.019676,0.276371,0.014837,27.814635,0.0,0.000052,-0.005319,-0.007203,0.000023,-0.014132,-0.016597,0.000219,0.000453,0.003134,-0.021586,false
-0.021822,0.016771,0.014466,0.016913,-0.043591,-0.006715,0.018412,0.032241,0.018583,-0.44673,-0.006715,0.031724,0.026761,0.032233,-0.601101,-0.006715,-0.022722,0.027153,-0.46875,-0.058148,27.792813,0.0,0.000951,0.009748,0.013117,-0.000631,-0.014402,-0.016084,0.000245,0.000599,0.000863,0.004667,true
0.004656,0.00712,0.013257,0.007145,0.29953,-0.003474,-0.002025,0.031282,-0.002023,-0.008929,-0.003474,0.025838,0.026692,0.026174,-0.140579,-0.003474,0.000507,0.029561,-0.068571,0.415113,27.797469,0.0,0.001395,-0.000042,-0.000655,0.003971,-0.000279,-0.003752,0.000095,-0.000063,0.000699,0.008784,true
0.008745,0.008914,0.013463,0.008954,0.014591,0.003489,-0.035045,0.028078,-0.034438,-0.149681,0.003489,-0.008088,0.025013,-0.008056,-0.33215,0.003489,0.011775,0.015991,0.431937,-0.264403,27.806214,0.0,0.000128,-0.001309,-0.002905,0.000196,-0.004203,-0.008308,0.000121,-0.000967,-0.000202,0.008791,true
0.008753,0.012611,0.013779,0.012691,0.108363,0.009708,0.05274,0.010343,0.054156,0.139535,0.009708,0.014546,0.024888,0.014653,-0.145985,0.009708,0.017394,0.022657,0.478022,0.148061,27.814967,0.0,0.000948,0.001221,-0.001278,0.001493,0.001443,-0.003633,0.000175,0.00056,0.000365,0.006806,true
0.006782,0.007115,0.013019,0.00714,0.082324,0.01505,0.046485,0.01001,0.047582,0.183791,0.01505,0.013528,0.024874,0.01362,-0.100949,0.01505,0.009067,0.020196,0.361224,0.030597,27.821749,0.0,0.000558,0.001247,-0.000685,0.001072,0.00184,-0.002511,0.000093,0.000476,0.000339,0.009562,true


In [153]:
SILVER_CSV_DIR = os.path.join('ETL', 'data', 'silver', 'csv')
os.makedirs(SILVER_CSV_DIR, exist_ok=True)
filepath = os.path.join(SILVER_CSV_DIR, 'silver_test.csv')
lf.head(50).collect().write_csv(filepath)